In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURACIÓN DE RUTAS

ruta_casos = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion"
ruta_base = os.path.join(ruta_casos, "Archivos_Unidos", "archivos por replica")

archivo_det_cen = os.path.join(ruta_base, "Detalle_centralizado_por_replica.csv")
archivo_det_dec = os.path.join(ruta_base, "Detalle_descentralizado_por_replica.csv")


# 2. CARGA DE DATOS

print("📥 Cargando datos de Detalle...")

def leer_csv(ruta):
    try:
        df = pd.read_csv(ruta, sep=';')
        if len(df.columns) < 3: df = pd.read_csv(ruta, sep=',')
        return df
    except: return pd.DataFrame()

df_det_cen = leer_csv(archivo_det_cen)
df_det_dec = leer_csv(archivo_det_dec)

df_det_cen['Escenario'] = 'Centralizado'
df_det_dec['Escenario'] = 'Descentralizado'
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)
df_det['Sede_Origen'] = df_det['Sede_Origen'].astype(str)


# 3. CÁLCULOS 

# 63156 = Grande (5), 53786 = Pequeña (2)
def cap_recursos(sede): return 5 if '63156' in str(sede) else 2

df_det['Capacidad'] = df_det['Sede_Origen'].apply(cap_recursos)

# Utilización de Cocina
if 'Cocineros_Libres_Al_Llegar' in df_det.columns:
    df_det['Utilizacion_Cocina'] = (1 - (df_det['Cocineros_Libres_Al_Llegar'] / df_det['Capacidad'])) * 100

# Pedidos por Moto (Productividad)
if 'Total_Pedidos' in df_det.columns:
    df_det['Pedidos_Por_Moto'] = df_det['Total_Pedidos'] / df_det['Capacidad']


# 4. CONFIGURACIÓN VISUAL 

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.color'] = '#cccccc'

color_desc = '#A62A2A' # Rojo oscuro (Descentralizado)
color_cent = '#1F4E79' # Azul marino (Centralizado)

def grafico_barras_(df, x_col, y_col, titulo, ylabel, filename):
    if y_col not in df.columns:
        print(f"⚠️ La columna '{y_col}' no está en los datos. Saltando gráfico.")
        return

    plt.figure(figsize=(8, 5.5))
    agrupado = df.groupby([x_col, 'Escenario'])[y_col].agg(['mean', 'std', 'count']).reset_index()
    # Calcular intervalo de confianza al 95%
    agrupado['ci'] = 1.96 * (agrupado['std'] / np.sqrt(agrupado['count']))
    
    sedes = sorted(agrupado[x_col].unique())
    x = np.arange(len(sedes))
    ancho = 0.35
    
    desc_data = agrupado[agrupado['Escenario'] == 'Descentralizado']
    plt.bar(x - ancho/2, desc_data['mean'], ancho, yerr=desc_data['ci'], label='Descentralizado', color=color_desc, capsize=5, edgecolor='black', linewidth=1)
    
    cent_data = agrupado[agrupado['Escenario'] == 'Centralizado']
    plt.bar(x + ancho/2, cent_data['mean'], ancho, yerr=cent_data['ci'], label='Centralizado', color=color_cent, capsize=5, edgecolor='black', linewidth=1)
    
    plt.title(titulo, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel(ylabel, fontsize=11)
    plt.xlabel("Sede", fontsize=11)
    plt.xticks(x, sedes)
    plt.legend(frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_base, filename), dpi=300, bbox_inches='tight')
    plt.close()

print("🎨 Generando los 4 gráficos")

# 1. Utilización de Cocina
grafico_barras_(df_det, 'Sede_Origen', 'Utilizacion_Cocina', 
                      "Utilización de Operarios de Cocina", "Porcentaje de Utilización (%)", 
                      "CASO2_grafico_utilizacion_cocina.png")

# 2. Productividad (Pedidos por Moto)
grafico_barras_(df_det, 'Sede_Origen', 'Pedidos_Por_Moto', 
                      "Productividad: Pedidos Atendidos por Moto", "Cantidad de Pedidos por Turno", 
                      "CASO2_grafico_productividad.png")

# 3. Espera por Moto (Cuello de Botella)
grafico_barras_(df_det, 'Sede_Origen', 'Espera_Moto_Lista_Min', 
                      "Tiempo Promedio de Espera por Repartidor", "Minutos de Espera", 
                      "CASO2_grafico_espera_moto.png")

# 4. Cola de Despacho (Saturación del sistema)
grafico_barras_(df_det, 'Sede_Origen', 'Cola_Despacho_En_Llegada', 
                     "Tamaño de la Cola de Despacho", "Cantidad de Pedidos en Cola", 
                      "CASO2_grafico_cola_despacho.png")

print("✅ ¡Los 4 gráficos están listos ")

📥 Cargando datos de Detalle...
🎨 Generando los 4 gráficos operativos adicionales...
⚠️ La columna 'Pedidos_Por_Moto' no está en los datos. Saltando gráfico.
✅ ¡Los 4 gráficos operativos adicionales están listos en tu carpeta!


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURACIÓN DE RUTAS

ruta_casos = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion"
ruta_base = os.path.join(ruta_casos, "Archivos_Unidos", "archivos por replica")

archivo_det_cen = os.path.join(ruta_base, "Detalle_centralizado_por_replica.csv")
archivo_det_dec = os.path.join(ruta_base, "Detalle_descentralizado_por_replica.csv")


# 2. CARGA DE DATOS

print("📥 Cargando datos de Detalle...")

def leer_csv(ruta):
    try:
        df = pd.read_csv(ruta, sep=';')
        if len(df.columns) < 3: df = pd.read_csv(ruta, sep=',')
        return df
    except: return pd.DataFrame()

df_det_cen = leer_csv(archivo_det_cen)
df_det_dec = leer_csv(archivo_det_dec)

df_det_cen['Escenario'] = 'Centralizado'
df_det_dec['Escenario'] = 'Descentralizado'
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)
df_det['Sede_Origen'] = df_det['Sede_Origen'].astype(str)


# 3. CÁLCULOS 

# 63156 = Grande (5), 53786 = Pequeña (2)
def cap_recursos(sede): return 5 if '63156' in str(sede) else 2

df_det['Capacidad'] = df_det['Sede_Origen'].apply(cap_recursos)

# Utilización de Cocina
if 'Cocineros_Libres_Al_Llegar' in df_det.columns:
    df_det['Utilizacion_Cocina'] = (1 - (df_det['Cocineros_Libres_Al_Llegar'] / df_det['Capacidad'])) * 100

# Pedidos por Moto (Productividad)
if 'Total_Pedidos' in df_det.columns:
    df_det['Pedidos_Por_Moto'] = df_det['Total_Pedidos'] / df_det['Capacidad']


# 4. CONFIGURACIÓN VISUAL 

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.color'] = '#cccccc'

color_desc = '#A62A2A' # Rojo oscuro (Descentralizado)
color_cent = '#1F4E79' # Azul marino (Centralizado)

def grafico_barras_(df, x_col, y_col, titulo, ylabel, filename):
    if y_col not in df.columns:
        print(f"⚠️ La columna '{y_col}' no está en los datos. Saltando gráfico.")
        return

    plt.figure(figsize=(8, 5.5))
    agrupado = df.groupby([x_col, 'Escenario'])[y_col].agg(['mean', 'std', 'count']).reset_index()
    # Calcular intervalo de confianza al 95%
    agrupado['ci'] = 1.96 * (agrupado['std'] / np.sqrt(agrupado['count']))
    
    sedes = sorted(agrupado[x_col].unique())
    x = np.arange(len(sedes))
    ancho = 0.35
    
    desc_data = agrupado[agrupado['Escenario'] == 'Descentralizado']
    plt.bar(x - ancho/2, desc_data['mean'], ancho, yerr=desc_data['ci'], label='Descentralizado', color=color_desc, capsize=5, edgecolor='black', linewidth=1)
    
    cent_data = agrupado[agrupado['Escenario'] == 'Centralizado']
    plt.bar(x + ancho/2, cent_data['mean'], ancho, yerr=cent_data['ci'], label='Centralizado', color=color_cent, capsize=5, edgecolor='black', linewidth=1)
    
    plt.title(titulo, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel(ylabel, fontsize=11)
    plt.xlabel("Sede", fontsize=11)
    plt.xticks(x, sedes)
    plt.legend(frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_base, filename), dpi=300, bbox_inches='tight')
    plt.close()

print("🎨 Generando los 4 gráficos ..")

# 1. Utilización de Cocina
grafico_barras_(df_det, 'Sede_Origen', 'Utilizacion_Cocina', 
                      "Utilización de Operarios de Cocina", "Porcentaje de Utilización (%)", 
                      "CASO2_grafico_utilizacion_cocina.png")

# 2. Productividad (Pedidos por Moto)
grafico_barras_(df_det, 'Sede_Origen', 'Pedidos_Por_Moto', 
                      "Productividad: Pedidos Atendidos por Moto", "Cantidad de Pedidos por Turno", 
                      "CASO2_grafico_productividad.png")

# 3. Espera por Moto (Cuello de Botella)
grafico_barras_(df_det, 'Sede_Origen', 'Espera_Moto_Lista_Min', 
                      "Tiempo Promedio de Espera por Repartidor", "Minutos de Espera", 
                      "CASO2_grafico_espera_moto.png")

# 4. Cola de Despacho (Saturación del sistema)
grafico_barras_(df_det, 'Sede_Origen', 'Cola_Despacho_En_Llegada', 
                      "Tamaño de la Cola de Despacho", "Cantidad de Pedidos en Cola", 
                      "CASO2_grafico_cola_despacho.png")

print("✅ ¡Los 4 gráficos están listos !")

📥 Cargando datos de Detalle...
🎨 Generando los 4 gráficos operativos adicionales...
⚠️ La columna 'Pedidos_Por_Moto' no está en los datos. Saltando gráfico.
✅ ¡Los 4 gráficos operativos adicionales están listos en tu carpeta!


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURACIÓN DE RUTAS Y NUEVA CARPETA

ruta_casos = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion"
ruta_lectura = os.path.join(ruta_casos, "Archivos_Unidos", "archivos por replica")

# Creamos la NUEVA CARPETA para guardar estos gráficos sin mezclar
ruta_salida = os.path.join(ruta_casos, "Archivos_Unidos", "graficos_operativos_caso2")
os.makedirs(ruta_salida, exist_ok=True)

archivo_det_cen = os.path.join(ruta_lectura, "Detalle_centralizado_por_replica.csv")
archivo_det_dec = os.path.join(ruta_lectura, "Detalle_descentralizado_por_replica.csv")


# 2. CARGA DE DATOS

print("📥 Cargando datos de Detalle...")

def leer_csv(ruta):
    try:
        df = pd.read_csv(ruta, sep=';')
        if len(df.columns) < 3: df = pd.read_csv(ruta, sep=',')
        return df
    except: return pd.DataFrame()

df_det_cen = leer_csv(archivo_det_cen)
df_det_dec = leer_csv(archivo_det_dec)

df_det_cen['Escenario'] = 'Centralizado'
df_det_dec['Escenario'] = 'Descentralizado'
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)
df_det['Sede_Origen'] = df_det['Sede_Origen'].astype(str)


# 3. CÁLCULOS 

# 63156 = Grande (5 recursos), 53786 = Pequeña (2 recursos)
def cap_recursos(sede): return 5 if '63156' in str(sede) else 2
df_det['Capacidad'] = df_det['Sede_Origen'].apply(cap_recursos)

# 3.1. Utilización y Productividad
if 'Cocineros_Libres_Al_Llegar' in df_det.columns:
    df_det['Utilizacion_Cocina'] = (1 - (df_det['Cocineros_Libres_Al_Llegar'] / df_det['Capacidad'])) * 100

if 'Total_Pedidos' in df_det.columns:
    df_det['Pedidos_Por_Moto'] = df_det['Total_Pedidos'] / df_det['Capacidad']

# 3.2. SLA a 23.5 minutos
df_sla = df_det.groupby(['Escenario', 'Sede_Origen'])['T_Ciclo_Total_Min'].apply(lambda x: (x <= 23.5).mean() * 100).reset_index(name='Cumplimiento_SLA')


# 4. CONFIGURACIÓN VISUAL 

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.color'] = '#cccccc'

color_desc = '#A62A2A' # Rojo oscuro
color_cent = '#1F4E79' # Azul marino

def grafico_barras(df, x_col, y_col, titulo, ylabel, filename):
    if y_col not in df.columns:
        print(f"⚠️ La columna '{y_col}' no está. Saltando gráfico.")
        return

    plt.figure(figsize=(8, 5.5))
    agrupado = df.groupby([x_col, 'Escenario'])[y_col].agg(['mean', 'std', 'count']).reset_index()
    agrupado['ci'] = 1.96 * (agrupado['std'] / np.sqrt(agrupado['count']))
    
    sedes = sorted(agrupado[x_col].unique())
    x = np.arange(len(sedes))
    ancho = 0.35
    
    desc_data = agrupado[agrupado['Escenario'] == 'Descentralizado']
    plt.bar(x - ancho/2, desc_data['mean'], ancho, yerr=desc_data['ci'], label='Descentralizado', color=color_desc, capsize=5, edgecolor='black', linewidth=1)
    
    cent_data = agrupado[agrupado['Escenario'] == 'Centralizado']
    plt.bar(x + ancho/2, cent_data['mean'], ancho, yerr=cent_data['ci'], label='Centralizado', color=color_cent, capsize=5, edgecolor='black', linewidth=1)
    
    plt.title(titulo, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel(ylabel, fontsize=11)
    plt.xlabel("Sede", fontsize=11)
    plt.xticks(x, sedes)
    plt.legend(frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_salida, filename), dpi=300, bbox_inches='tight')
    plt.close()

print("🎨 Generando gráficos en la nueva carpeta 'graficos_operativos_caso2'...")

# 1. Utilización de Cocina
grafico_barras(df_det, 'Sede_Origen', 'Utilizacion_Cocina', 
                      "Utilización de Operarios de Cocina", "Porcentaje de Utilización (%)", 
                      "CASO2_grafico_utilizacion_cocina.png")

# 2. Productividad
grafico_barras(df_det, 'Sede_Origen', 'Pedidos_Por_Moto', 
                      "Productividad: Pedidos Atendidos por Moto", "Cantidad de Pedidos por Turno", 
                      "CASO2_grafico_productividad.png")

# 3. Espera por Moto
grafico_barras(df_det, 'Sede_Origen', 'Espera_Moto_Lista_Min', 
                      "Tiempo Promedio de Espera por Repartidor", "Minutos de Espera", 
                      "CASO2_grafico_espera_moto.png")

# 4. Cola de Despacho
grafico_barras(df_det, 'Sede_Origen', 'Cola_Despacho_En_Llegada', 
                      "Tamaño de la Cola de Despacho", "Cantidad de Pedidos en Cola", 
                      "CASO2_grafico_cola_despacho.png")


# 5. GRÁFICO SLA (intento de ESTILO MANCUERNA / DUMBBELL) 

plt.figure(figsize=(8, 5))
df_pivot = df_sla.pivot(index='Sede_Origen', columns='Escenario', values='Cumplimiento_SLA').reset_index()

for i, row in df_pivot.iterrows():
    if 'Descentralizado' in row and 'Centralizado' in row:
        val_desc = row['Descentralizado']
        val_cent = row['Centralizado']
        plt.plot([val_desc, val_cent], [i, i], color='grey', linestyle='-', linewidth=2.5, zorder=1, alpha=0.7)
        plt.scatter(val_desc, i, color=color_desc, s=200, zorder=2, edgecolor='black')
        plt.scatter(val_cent, i, color=color_cent, s=200, zorder=2, edgecolor='black')
        plt.text(val_desc, i + 0.15, f"{val_desc:.1f}%", color=color_desc, ha='center', fontweight='bold', fontsize=11)
        plt.text(val_cent, i - 0.20, f"{val_cent:.1f}%", color=color_cent, ha='center', fontweight='bold', fontsize=11)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Descentralizado', markerfacecolor=color_desc, markersize=10, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', label='Centralizado', markerfacecolor=color_cent, markersize=10, markeredgecolor='black')
]
plt.yticks(range(len(df_pivot)), df_pivot['Sede_Origen'])
plt.xlabel("Probabilidad de cumplimiento (%)", fontsize=11)
plt.ylabel("Sede", fontsize=11)
plt.title("Cumplimiento de Nivel de Servicio (SLA < 23.5 min)", fontsize=14, fontweight='bold', pad=20)
plt.xlim(0, 105)
plt.legend(handles=legend_elements, frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(ruta_salida, "CASO2_grafico_SLA_Mancuerna.png"), dpi=300, bbox_inches='tight')
plt.close()


# 6. GRÁFICO CDF (FUNCIÓN DE DISTRIBUCIÓN ACUMULADA)

plt.figure(figsize=(8, 5.5))
plt.axvline(x=23.5, color='black', linestyle='--', linewidth=1.5, label='SLA Promesa (23.5 min)')

estilos_linea = {'Descentralizado': '-', 'Centralizado': '-'}
colores_linea = {'Descentralizado': color_desc, 'Centralizado': color_cent}

for escenario in ['Descentralizado', 'Centralizado']:
    datos = df_det[df_det["Escenario"] == escenario]["T_Ciclo_Total_Min"].dropna()
    p99 = np.percentile(datos, 99)
    datos = datos[datos <= p99]
    
    if len(datos) > 0:
        datos_ordenados = np.sort(datos)
        p = np.arange(len(datos_ordenados)) / (len(datos_ordenados) - 1) if len(datos_ordenados) > 1 else np.array([1.0])
        plt.plot(datos_ordenados, p, label=escenario, linewidth=2.5, 
                 linestyle=estilos_linea[escenario], color=colores_linea[escenario])

plt.title("Función de Distribución Acumulada (Lead Time)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Lead Time Promedio (minutos)", fontsize=11)
plt.ylabel("Probabilidad Acumulada (%)", fontsize=11)

yticks = plt.gca().get_yticks()
plt.gca().set_yticklabels([f'{int(y*100)}%' for y in yticks])

plt.xlim(15, np.percentile(df_det['T_Ciclo_Total_Min'].dropna(), 95))
plt.legend(frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.tight_layout()
plt.savefig(os.path.join(ruta_salida, "CASO2_grafico_CDF_leadtime.png"), dpi=300, bbox_inches='tight')
plt.close()

print(f"✅ ¡Todo listo! Revisa la nueva carpeta: {ruta_salida}")

📥 Cargando datos de Detalle...
🎨 Generando 6 gráficos en la nueva carpeta 'graficos_operativos_caso2'...
⚠️ La columna 'Pedidos_Por_Moto' no está. Saltando gráfico.
✅ ¡Todo listo! Revisa la nueva carpeta: C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion\Archivos_Unidos\graficos_operativos_caso2


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURACIÓN DE RUTAS

ruta_casos = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion"
ruta_lectura = os.path.join(ruta_casos, "Archivos_Unidos", "archivos por replica")
ruta_salida = os.path.join(ruta_casos, "Archivos_Unidos", "graficos_operativos_caso2")
os.makedirs(ruta_salida, exist_ok=True)

archivo_det_cen = os.path.join(ruta_lectura, "Detalle_centralizado_por_replica.csv")
archivo_det_dec = os.path.join(ruta_lectura, "Detalle_descentralizado_por_replica.csv")


# 2. CARGA DE DATOS

print("📥 Cargando datos...")
def leer_csv(ruta):
    try:
        df = pd.read_csv(ruta, sep=';')
        if len(df.columns) < 3: df = pd.read_csv(ruta, sep=',')
        return df
    except: return pd.DataFrame()

df_det_cen = leer_csv(archivo_det_cen)
df_det_dec = leer_csv(archivo_det_dec)
df_det_cen['Escenario'] = 'Centralizado'
df_det_dec['Escenario'] = 'Descentralizado'
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)
df_det['Sede_Origen'] = df_det['Sede_Origen'].astype(str)

# Cargar RESUMEN (Para Overtime y Total_Pedidos)
archivos_res_cen = glob.glob(os.path.join(ruta_casos, "caso centralizado", "*RESUMEN*.csv"))
archivos_res_dec = glob.glob(os.path.join(ruta_casos, "caso descentralizado", "*RESUMEN*.csv"))
dfs_res = []
for f in archivos_res_cen:
    df = leer_csv(f)
    df['Escenario'] = 'Centralizado'
    dfs_res.append(df)
for f in archivos_res_dec:
    df = leer_csv(f)
    df['Escenario'] = 'Descentralizado'
    dfs_res.append(df)
    
df_res = pd.concat(dfs_res, ignore_index=True) if dfs_res else pd.DataFrame()
if not df_res.empty: 
    df_res['Sede'] = df_res['Sede'].astype(str)
    # Cruzamos el Total de Pedidos hacia el detalle para poder calcular el kilometraje
    df_det = pd.merge(df_det, df_res[['Sede', 'Replica', 'Escenario', 'Total_Pedidos']], 
                      left_on=['Sede_Origen', 'Replica', 'Escenario'], 
                      right_on=['Sede', 'Replica', 'Escenario'], how='left')


# 3. CÁLCULOS 

# 63156 = Grande (5 recursos), 53786 = Pequeña (2 recursos)
def cap_recursos(sede): return 5 if '63156' in str(sede) else 2
df_det['Capacidad'] = df_det['Sede_Origen'].apply(cap_recursos)

if 'Motos_Libres_Al_Llegar' in df_det.columns:
    df_det['Utilizacion_Motos'] = (1 - (df_det['Motos_Libres_Al_Llegar'] / df_det['Capacidad'])) * 100
if 'Cocineros_Libres_Al_Llegar' in df_det.columns:
    df_det['Utilizacion_Cocina'] = (1 - (df_det['Cocineros_Libres_Al_Llegar'] / df_det['Capacidad'])) * 100

# Kilometraje Total
FACTOR_IDA_VUELTA = 2
if 'Distancia_KM' in df_det.columns and 'Total_Pedidos' in df_det.columns:
    df_det['Kilometraje_Total'] = df_det['Distancia_KM'] * df_det['Total_Pedidos'] * FACTOR_IDA_VUELTA


# 4. FUNCIONES VISUALES 

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.color'] = '#cccccc'

color_desc = '#A62A2A' # Rojo oscuro
color_cent = '#1F4E79' # Azul marino

def grafico_barras_(df, x_col, y_col, titulo, ylabel, filename):
    if y_col not in df.columns: return
    plt.figure(figsize=(8, 5.5))
    agrupado = df.groupby([x_col, 'Escenario'])[y_col].agg(['mean', 'std', 'count']).reset_index()
    agrupado['ci'] = 1.96 * (agrupado['std'] / np.sqrt(agrupado['count']))
    
    sedes = sorted(agrupado[x_col].unique())
    x = np.arange(len(sedes))
    ancho = 0.35
    
    desc_data = agrupado[agrupado['Escenario'] == 'Descentralizado']
    plt.bar(x - ancho/2, desc_data['mean'], ancho, yerr=desc_data['ci'], label='Descentralizado', color=color_desc, capsize=5, edgecolor='black', linewidth=1)
    
    cent_data = agrupado[agrupado['Escenario'] == 'Centralizado']
    plt.bar(x + ancho/2, cent_data['mean'], ancho, yerr=cent_data['ci'], label='Centralizado', color=color_cent, capsize=5, edgecolor='black', linewidth=1)
    
    plt.title(titulo, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel(ylabel, fontsize=11)
    plt.xlabel("Sede", fontsize=11)
    plt.xticks(x, sedes)
    plt.legend(frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_salida, filename), dpi=300, bbox_inches='tight')
    plt.close()

def grafico_mariposa(df, groupby_col, val_col, titulo, xlabel, filename):
    df_agrupado = df.groupby([groupby_col, 'Escenario'])[val_col].mean().reset_index()
    df_pivot = df_agrupado.pivot(index=groupby_col, columns='Escenario', values=val_col).fillna(0)
    
    sedes = df_pivot.index.tolist()
    y_pos = np.arange(len(sedes))
    val_desc = df_pivot['Descentralizado'].values
    val_cent = df_pivot['Centralizado'].values

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(y_pos, -val_desc, align='center', color=color_desc, edgecolor='black', label='Descentralizado', height=0.5)
    ax.barh(y_pos, val_cent, align='center', color=color_cent, edgecolor='black', label='Centralizado', height=0.5)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(sedes, fontweight='bold', fontsize=12)
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_title(titulo, fontweight='bold', pad=20, fontsize=15)

    ticks = ax.get_xticks()
    ax.set_xticklabels([str(abs(int(tick))) for tick in ticks])

    for i, (vd, vc) in enumerate(zip(val_desc, val_cent)):
        # Ajustamos el texto para que diga "X min" si es tiempo, o "X km" si no
        unidad = " min" if "tiempo" in titulo.lower() or "overtime" in titulo.lower() else ""
        ax.text(-vd - (vd*0.05), i, f"{vd:.1f}{unidad}", va='center', ha='right', color='black', fontweight='bold', fontsize=10)
        ax.text(vc + (vc*0.05), i, f"{vc:.1f}{unidad}", va='center', ha='left', color='black', fontweight='bold', fontsize=10)

    ax.axvline(0, color='black', linewidth=1.5)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=True)
    
    max_val = max(val_desc.max(), val_cent.max()) * 1.30
    ax.set_xlim(-max_val, max_val)

    plt.tight_layout()
    plt.savefig(os.path.join(ruta_salida, filename), dpi=300, bbox_inches='tight')
    plt.close()

print("🎨 Generando gráficos...")

# 1 a 5: Gráficos de Barras
grafico_barras(df_det, 'Sede_Origen', 'Cola_Despacho_En_Llegada', "Tamaño Promedio de la Cola de Despacho", "Pedidos en Cola", "CASO2_grafico_cola_despacho.png")
grafico_barras(df_det, 'Sede_Origen', 'Espera_Moto_Lista_Min', "Tiempo Promedio de Espera por Repartidor", "Minutos", "CASO2_grafico_espera_moto.png")
grafico_barras(df_det, 'Sede_Origen', 'Utilizacion_Cocina', "Utilización de Operarios de Cocina", "Utilización (%)", "CASO2_grafico_utilizacion_cocina.png")
grafico_barras(df_det, 'Sede_Origen', 'Utilizacion_Motos', "Utilización de la Flota de Motos", "Utilización (%)", "CASO2_grafico_utilizacion_motos.png")
grafico_barras(df_det, 'Sede_Origen', 'Kilometraje_Total', "Kilometraje Total de la Flota (Ida y Vuelta)", "Kilómetros Recorridos", "CASO2_grafico_kilometraje.png")

# 6. Mariposa para Overtime
if not df_res.empty and 'Overtime_Min' in df_res.columns:
    grafico_mariposa(df_res, 'Sede', 'Overtime_Min', "Promedio de Overtime por Sede", "Minutos de Overtime", "CASO2_grafico_overtime_mariposa.png")

# 7. Mariposa para Lead Time (T_Ciclo_Total_Min)
if 'T_Ciclo_Total_Min' in df_det.columns:
    grafico_mariposa(df_det, 'Sede_Origen', 'T_Ciclo_Total_Min', "Tiempo de Ciclo Promedio (Lead Time)", "Minutos", "CASO2_grafico_leadtime_mariposa.png")

print("✅ ¡Los 7 gráficos  están listos!")

📥 Cargando datos...
🎨 Generando tu set final de gráficos...
✅ ¡Los 7 gráficos definitivos están listos!


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURACIÓN DE RUTAS

ruta_casos = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion"
ruta_lectura = os.path.join(ruta_casos, "Archivos_Unidos", "archivos por replica")
ruta_salida = os.path.join(ruta_casos, "Archivos_Unidos", "graficos_operativos_caso2")
os.makedirs(ruta_salida, exist_ok=True)

archivo_det_cen = os.path.join(ruta_lectura, "Detalle_centralizado_por_replica.csv")
archivo_det_dec = os.path.join(ruta_lectura, "Detalle_descentralizado_por_replica.csv")


# 2. CARGA DE DATOS

print("📥 Cargando datos...")
def leer_csv(ruta):
    try:
        df = pd.read_csv(ruta, sep=';')
        if len(df.columns) < 3: df = pd.read_csv(ruta, sep=',')
        return df
    except: return pd.DataFrame()

df_det_cen = leer_csv(archivo_det_cen)
df_det_dec = leer_csv(archivo_det_dec)
df_det_cen['Escenario'] = 'Centralizado'
df_det_dec['Escenario'] = 'Descentralizado'
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)
df_det['Sede_Origen'] = df_det['Sede_Origen'].astype(str)

# Cargar RESUMEN (Para obtener el Total_Pedidos)
archivos_res_cen = glob.glob(os.path.join(ruta_casos, "caso centralizado", "*RESUMEN*.csv"))
archivos_res_dec = glob.glob(os.path.join(ruta_casos, "caso descentralizado", "*RESUMEN*.csv"))
dfs_res = []
for f in archivos_res_cen:
    df = leer_csv(f)
    df['Escenario'] = 'Centralizado'
    dfs_res.append(df)
for f in archivos_res_dec:
    df = leer_csv(f)
    df['Escenario'] = 'Descentralizado'
    dfs_res.append(df)
    
df_res = pd.concat(dfs_res, ignore_index=True) if dfs_res else pd.DataFrame()
if not df_res.empty: 
    df_res['Sede'] = df_res['Sede'].astype(str)
    # Cruzamos el Total de Pedidos hacia el detalle
    df_det = pd.merge(df_det, df_res[['Sede', 'Replica', 'Escenario', 'Total_Pedidos']], 
                      left_on=['Sede_Origen', 'Replica', 'Escenario'], 
                      right_on=['Sede', 'Replica', 'Escenario'], how='left')


# 3. ESTILO VISUAL 

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.color'] = '#cccccc'

color_desc = '#A62A2A' # Rojo oscuro
color_cent = '#1F4E79' # Azul marino

print("🎨 Generando los  gráficos .")

-
# GRÁFICO 1: NÚMERO DE PEDIDOS ATENDIDOS (BARRAS)

if 'Total_Pedidos' in df_det.columns:
    plt.figure(figsize=(8, 5.5))
    agrupado = df_det.groupby(['Sede_Origen', 'Escenario'])['Total_Pedidos'].agg(['mean', 'std', 'count']).reset_index()
    agrupado['ci'] = 1.96 * (agrupado['std'] / np.sqrt(agrupado['count']))
    
    sedes = sorted(agrupado['Sede_Origen'].unique())
    x = np.arange(len(sedes))
    ancho = 0.35
    
    desc_data = agrupado[agrupado['Escenario'] == 'Descentralizado']
    plt.bar(x - ancho/2, desc_data['mean'], ancho, yerr=desc_data['ci'], label='Descentralizado', color=color_desc, capsize=5, edgecolor='black', linewidth=1)
    
    cent_data = agrupado[agrupado['Escenario'] == 'Centralizado']
    plt.bar(x + ancho/2, cent_data['mean'], ancho, yerr=cent_data['ci'], label='Centralizado', color=color_cent, capsize=5, edgecolor='black', linewidth=1)
    
    plt.title("Total de Pedidos Atendidos por Sede", fontsize=14, fontweight='bold', pad=15)
    plt.ylabel("Cantidad de Pedidos", fontsize=11)
    plt.xlabel("Sede", fontsize=11)
    plt.xticks(x, sedes)
    plt.legend(frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_salida, "CASO2_grafico_pedidos_atendidos.png"), dpi=300, bbox_inches='tight')
    plt.close()


# GRÁFICO 2: HISTOGRAMA DE DISTRIBUCIÓN DE TIEMPOS (LEAD TIME)

if 'T_Ciclo_Total_Min' in df_det.columns:
    # Vamos a crear 2 subgráficos (uno para cada sede) para que el histograma sea clarísimo
    sedes = sorted(df_det['Sede_Origen'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
    
    for i, sede in enumerate(sedes):
        ax = axes[i]
        datos_desc = df_det[(df_det['Sede_Origen'] == sede) & (df_det['Escenario'] == 'Descentralizado')]['T_Ciclo_Total_Min'].dropna()
        datos_cent = df_det[(df_det['Sede_Origen'] == sede) & (df_det['Escenario'] == 'Centralizado')]['T_Ciclo_Total_Min'].dropna()
        
        # Filtramos el top 1% de casos atípicos extremos para que la campana se vea bien
        limite = np.percentile(df_det['T_Ciclo_Total_Min'].dropna(), 99)
        datos_desc = datos_desc[datos_desc < limite]
        datos_cent = datos_cent[datos_cent < limite]

        # Dibujamos los dos histogramas superpuestos
        ax.hist(datos_desc, bins=20, color=color_desc, alpha=0.6, edgecolor='black', linewidth=0.8, density=True, label='Descentralizado')
        ax.hist(datos_cent, bins=20, color=color_cent, alpha=0.6, edgecolor='black', linewidth=0.8, density=True, label='Centralizado')
        
        # Línea de SLA
        ax.axvline(23.5, color='black', linestyle='--', linewidth=1.5, label='SLA (23.5 min)')
        
        ax.set_title(f"Distribución en Sede {sede}", fontweight='bold', fontsize=12)
        ax.set_xlabel("Tiempo Promedio de Entrega (min)", fontsize=11)
        if i == 0:
            ax.set_ylabel("Densidad de Probabilidad", fontsize=11)
            
    # Leyenda global
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3, frameon=True)
    
    plt.suptitle("Histograma: Distribución del Tiempo de Ciclo (Lead Time)", fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_salida, "CASO2_grafico_histograma_tiempos.png"), dpi=300, bbox_inches='tight')
    plt.close()

print("✅ ¡Los 2 gráficos nuevos están listos en tu carpeta 'graficos_operativos_caso2'!")

📥 Cargando datos...
🎨 Generando los 2 gráficos solicitados...
✅ ¡Los 2 gráficos nuevos están listos en tu carpeta 'graficos_operativos_caso2'!


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')


# 1. CONFIGURACIÓN DE RUTAS

ruta_casos = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion"
ruta_lectura = os.path.join(ruta_casos, "Archivos_Unidos", "archivos por replica")
ruta_salida = os.path.join(ruta_casos, "Archivos_Unidos", "graficos_operativos_caso2")
os.makedirs(ruta_salida, exist_ok=True)

archivo_det_cen = os.path.join(ruta_lectura, "Detalle_centralizado_por_replica.csv")
archivo_det_dec = os.path.join(ruta_lectura, "Detalle_descentralizado_por_replica.csv")


# 2. CARGA DE DATOS

print("📥 Cargando datos...")
def leer_csv(ruta):
    try:
        df = pd.read_csv(ruta, sep=';')
        if len(df.columns) < 3: df = pd.read_csv(ruta, sep=',')
        return df
    except: return pd.DataFrame()

df_det_cen = leer_csv(archivo_det_cen)
df_det_dec = leer_csv(archivo_det_dec)
df_det_cen['Escenario'] = 'Centralizado'
df_det_dec['Escenario'] = 'Descentralizado'
df_det = pd.concat([df_det_cen, df_det_dec], ignore_index=True)
df_det['Sede_Origen'] = df_det['Sede_Origen'].astype(str)


# 3. ESTILO VISUAL 

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
color_desc = '#A62A2A' # Rojo oscuro
color_cent = '#1F4E79' # Azul marino

print("🎨 Generando  Correlación...")


# GRÁFICO 1: DISPERSIÓN (SCATTER PLOT) - DISTANCIA VS LEAD TIME

if 'Distancia_KM' in df_det.columns and 'T_Ciclo_Total_Min' in df_det.columns:
    plt.figure(figsize=(9, 6))
    
    # Filtramos extremos atípicos (el 1% más raro) solo para que el gráfico se vea limpio
    q_dist = df_det['Distancia_KM'].quantile(0.99)
    q_time = df_det['T_Ciclo_Total_Min'].quantile(0.99)
    df_clean = df_det[(df_det['Distancia_KM'] <= q_dist) & (df_det['T_Ciclo_Total_Min'] <= q_time)]

    # Puntos Descentralizado
    plt.scatter(df_clean[df_clean['Escenario']=='Descentralizado']['Distancia_KM'], 
                df_clean[df_clean['Escenario']=='Descentralizado']['T_Ciclo_Total_Min'], 
                color=color_desc, alpha=0.5, label='Descentralizado', edgecolors='white')
    
    # Puntos Centralizado
    plt.scatter(df_clean[df_clean['Escenario']=='Centralizado']['Distancia_KM'], 
                df_clean[df_clean['Escenario']=='Centralizado']['T_Ciclo_Total_Min'], 
                color=color_cent, alpha=0.5, label='Centralizado', edgecolors='white')
    
    # Líneas de Tendencia
    z_desc = np.polyfit(df_clean[df_clean['Escenario']=='Descentralizado']['Distancia_KM'], df_clean[df_clean['Escenario']=='Descentralizado']['T_Ciclo_Total_Min'], 1)
    p_desc = np.poly1d(z_desc)
    plt.plot(df_clean[df_clean['Escenario']=='Descentralizado']['Distancia_KM'], p_desc(df_clean[df_clean['Escenario']=='Descentralizado']['Distancia_KM']), color='darkred', linestyle='--', linewidth=2.5)

    z_cent = np.polyfit(df_clean[df_clean['Escenario']=='Centralizado']['Distancia_KM'], df_clean[df_clean['Escenario']=='Centralizado']['T_Ciclo_Total_Min'], 1)
    p_cent = np.poly1d(z_cent)
    plt.plot(df_clean[df_clean['Escenario']=='Centralizado']['Distancia_KM'], p_cent(df_clean[df_clean['Escenario']=='Centralizado']['Distancia_KM']), color='darkblue', linestyle='--', linewidth=2.5)

    plt.title('Correlación: Distancia Recorrida vs Tiempo de Entrega', fontweight='bold', fontsize=14, pad=15)
    plt.xlabel('Distancia Promedio por Réplica (KM)', fontsize=11)
    plt.ylabel('Tiempo de Ciclo Promedio (Minutos)', fontsize=11)
    plt.legend(frameon=True, loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Quitar bordes innecesarios
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_salida, 'CASO2_grafico_dispersion_distancia.png'), dpi=300, bbox_inches='tight')
    plt.close()


# GRÁFICO 2: MAPA DE CALOR (HEATMAP) DE CORRELACIONES

variables_corr = ['T_Ciclo_Total_Min', 'Distancia_KM', 'Motos_Libres_Al_Llegar', 'Espera_Moto_Lista_Min', 'Cola_Despacho_En_Llegada']
# Dejar solo las columnas que existan en el archivo
variables_corr = [v for v in variables_corr if v in df_det.columns]

if len(variables_corr) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Limpiamos los nombres para que se vean bonitos en el gráfico
    nombres_limpios = [v.replace('_', ' ').replace('Min', '').replace('KM', '').strip() for v in variables_corr]
    
    for i, esc in enumerate(['Descentralizado', 'Centralizado']):
        corr_matrix = df_det[df_det['Escenario'] == esc][variables_corr].corr()
        
        # Generar el Heatmap usando Seaborn
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", 
                    xticklabels=nombres_limpios, yticklabels=nombres_limpios, 
                    ax=axes[i], vmin=-1, vmax=1, cbar=(i==1), 
                    linewidths=0.5, linecolor='white')
        
        axes[i].set_title(f'Matriz de Correlación\n({esc})', fontweight='bold', pad=15, fontsize=13)
        
        # Rotar etiquetas para que se lean bien
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(os.path.join(ruta_salida, 'CASO2_grafico_heatmap_correlacion.png'), dpi=300, bbox_inches='tight')
    plt.close()

print("✅ ¡Los 2 gráficos de correlación están listos en tu carpeta 'graficos_operativos_caso2'!")

📥 Cargando datos...
🎨 Generando Análisis de Correlación...
✅ ¡Los 2 gráficos de correlación están listos en tu carpeta 'graficos_operativos_caso2'!


In [1]:
import pandas as pd
import numpy as np
from scipy import stats


# CONFIGURACIÓN - AJUSTA ESTA RUTA

ruta_cen = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion\Archivos_Unidos\archivos por replica\Detalle_centralizado_por_replica.csv"
ruta_dec = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion\Archivos_Unidos\archivos por replica\Detalle_descentralizado_por_replica.csv"
ruta_salida = r"C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Segundo caso de simulacion\Archivos_Unidos\archivos por replica"


# CARGA

df_cen = pd.read_csv(ruta_cen, sep=';')
df_dec = pd.read_csv(ruta_dec, sep=';')


# TABLA 1: ESTADÍSTICAS DESCRIPTIVAS + IC95%

def ic95(datos):
    n = len(datos)
    media = datos.mean()
    se = datos.std() / np.sqrt(n)
    return media, media - 1.96*se, media + 1.96*se

filas = []
for esc, df in [('Centralizado', df_cen), ('Descentralizado', df_dec)]:
    for sede in sorted(df['Sede_Origen'].unique()):
        d = df[df['Sede_Origen'] == sede]['T_Ciclo_Total_Min']
        media, ic_inf, ic_sup = ic95(d)
        filas.append({
            'Escenario':        esc,
            'Sede':             sede,
            'n':                len(d),
            'Media':            round(media, 2),
            'Mediana':          round(d.median(), 2),
            'Desv. Std':        round(d.std(), 2),
            'Mín':              round(d.min(), 2),
            'Máx':              round(d.max(), 2),
            'IC 95% Inferior':  round(ic_inf, 2),
            'IC 95% Superior':  round(ic_sup, 2),
        })

df_tabla1 = pd.DataFrame(filas)
print("TABLA 1 — Estadísticas Descriptivas del Lead Time (minutos)")
print(df_tabla1.to_string(index=False))


# TABLA 2: MANN-WHITNEY LEAD TIME

filas_mw = []
for sede in sorted(df_cen['Sede_Origen'].unique()):
    dec = df_dec[df_dec['Sede_Origen'] == sede]['T_Ciclo_Total_Min']
    cen = df_cen[df_cen['Sede_Origen'] == sede]['T_Ciclo_Total_Min']
    u, p = stats.mannwhitneyu(dec, cen, alternative='two-sided')
    mejora = ((cen.median() - dec.median()) / abs(cen.median())) * 100
    filas_mw.append({
        'Sede':             sede,
        'Mediana Dec.':     round(dec.median(), 2),
        'Mediana Cen.':     round(cen.median(), 2),
        'Estadístico U':    f"{u:.0f}",
        'P-Value':          f"{p:.2e}",
        'Reducción (%)':    f"{mejora:+.1f}%",
        'Decisión':         "Rechazar H0 ✅" if p < 0.05 else "No rechazar H0",
    })

df_tabla2 = pd.DataFrame(filas_mw)
print("\nTABLA 2 — Prueba U de Mann-Whitney para Lead Time")
print("H0: No hay diferencia en medianas | α = 0.05")
print(df_tabla2.to_string(index=False))


# EXPORTAR

import os
df_tabla1.to_csv(os.path.join(ruta_salida, "CASO2_descriptivas_leadtime.csv"), index=False, sep=';')
df_tabla2.to_csv(os.path.join(ruta_salida, "CASO2_mannwhitney_leadtime.csv"), index=False, sep=';')
print("\n✅ Tablas exportadas a CSV en:", ruta_salida)

TABLA 1 — Estadísticas Descriptivas del Lead Time (minutos)
      Escenario  Sede   n  Media  Mediana  Desv. Std   Mín    Máx  IC 95% Inferior  IC 95% Superior
   Centralizado 53786 500  85.31    83.87      25.26 33.75 175.99            83.09            87.52
   Centralizado 63156 500  22.04    22.03       0.47 20.63  23.30            22.00            22.08
Descentralizado 53786 500  37.38    37.35       0.72 35.53  39.59            37.32            37.45
Descentralizado 63156 500  22.50    22.50       0.05 22.33  22.62            22.49            22.50

TABLA 2 — Prueba U de Mann-Whitney para Lead Time
H0: No hay diferencia en medianas | α = 0.05
 Sede  Mediana Dec.  Mediana Cen. Estadístico U   P-Value Reducción (%)      Decisión
53786         37.35         83.87          1297 1.35e-161        +55.5% Rechazar H0 ✅
63156         22.50         22.03        205844  3.97e-70         -2.1% Rechazar H0 ✅

✅ Tablas exportadas a CSV en: C:\Users\Tobias\Desktop\tesis simulacion geoespacial\Se